# Scored models — revenue forecast & AR late-payment risk

Two graded models (SKILL.md §11), trained from the stored synthetic CSVs in
`backend/data/` — **not** regenerated here, so the dataset a model saw is an
inspectable file. Run `python -m ml.generate_training_data` once to create them.

Everything below calls the same code the app serves: `ml/train_models.py` (fit and
evaluate) and `app/ml.py` (inference), so no number here is a second implementation.

In [ ]:
import sys, json, pathlib

# Resolve backend/ whether the notebook is run from the repo root or backend/.
HERE = pathlib.Path.cwd()
BACKEND = HERE if (HERE / "data" / "ar_training.csv").exists() else HERE / "backend"
if not (BACKEND / "data" / "ar_training.csv").exists():
    BACKEND = HERE.parent / "backend"
DATA = BACKEND / "data"
sys.path.insert(0, str(BACKEND))
print("data dir:", DATA.resolve())

from ml.train_models import train_all, AR_NUMERIC, REVENUE_FEATURES, EXPECTED_F1_CEILING
print("AR features:", AR_NUMERIC, "+ one-hot categories")
print("Revenue features:", REVENUE_FEATURES)
print("Ceiling agreed BEFORE training (F1):", EXPECTED_F1_CEILING)

## 1. Look at the stored training data first

In [ ]:
import pandas as pd
ar = pd.read_csv(str(DATA / "ar_training.csv"))
rev = pd.read_csv(str(DATA / "revenue_training.csv"))
print("AR rows:", len(ar),
      "| distinct customers:", ar.vendor_or_customer.nunique(),
      "| late rate:", f"{ar.paid_late.mean():.1%}")
print("Revenue months:", len(rev),
      "| range:", f"${rev.revenue.min():,.0f}-${rev.revenue.max():,.0f}")
ar.head(3)

## 2. Train + evaluate both models (chronological split, baseline first)

Writes `models/*.joblib` and `models/metrics.json` — the artifacts the app loads.

In [ ]:
metrics = train_all()

## 3. AR late-payment risk — baseline vs model, with the confusion matrix

Accuracy alone is misleading here: "always on time" scores high while catching
**zero** late payments.

In [ ]:
ar_m = metrics["ar_risk"]
print(f"Baseline ({ar_m['baseline']['name']}): accuracy {ar_m['baseline']['accuracy']:.3f}"
      f" — caught {ar_m['baseline']['late_caught']}/{ar_m['confusion_matrix']['late_actual']} late")
print(f"Model: accuracy {ar_m['model_metrics']['accuracy']:.3f}"
      f" precision {ar_m['model_metrics']['precision']:.3f}"
      f" recall {ar_m['model_metrics']['recall']:.3f}"
      f" F1 {ar_m['model_metrics']['f1']:.3f} @ threshold {ar_m['threshold']}")

cm = ar_m["confusion_matrix"]
print()
print("Confusion matrix  rows = actual, cols = predicted [on_time, late]")
print(pd.DataFrame(cm["matrix"], index=cm["labels"], columns=cm["labels"]))
print(f"Raw counts: caught {cm['late_caught']}/{cm['late_actual']} actual-late rows;"
      f" {cm['false_positive']} false alarms; {cm['false_negative']} missed")
print("Ceiling check:", ar_m["ceiling_check"])

## 4. Revenue forecast — persistence baseline vs regression

In [ ]:
rev_m = metrics["revenue"]
print(f"Baseline ({rev_m['baseline']['name']}): MAE ${rev_m['baseline']['mae']:,.0f}")
print(f"Model ({rev_m['model']}): MAE ${rev_m['model_metrics']['mae']:,.0f}"
      f" ({rev_m['improvement_pct']}% lower error)")
print("Coefficients:", rev_m["coefficients"])

## 5. Failure cases (required by the brief)

One concrete case per model that the model got clearly wrong, plus a hypothesis.

In [ ]:
print("AR failure:", json.dumps(metrics["ar_risk"]["failure_case"], indent=2))
print()
print("Revenue failure:", json.dumps(metrics["revenue"]["failure_case"], indent=2))

## 6. Same functions the app calls

`app/ml.py` loads these artifacts at request time, so the dashboard and the
advisor quote exactly the numbers above.

In [ ]:
from datetime import date

from app import ml
from app.models import Transaction

# Pinned, not date.today(). The write-up's worked example is "as of 20 Nov 2026";
# asking the clock would make these numbers drift a little every day, so the
# scenario in the submission could not be reproduced later.
AS_OF = date(2026, 11, 20)

txs = [
    Transaction(type="income", amount=8000, transaction_date="2026-11-05", status="paid"),
    Transaction(type="ar", amount=6400, category="Customer Invoice",
                vendor_or_customer="Mesa Retail", transaction_date="2026-09-01",
                due_date="2026-09-15", status="overdue"),
    Transaction(type="ar", amount=5200, category="Customer Invoice",
                vendor_or_customer="Oak & Pine Co.", transaction_date="2026-11-01",
                due_date="2026-11-28", status="pending"),
]

summary = ml.ar_risk_summary(txs, today=AS_OF)
print(f"scoring moment: {AS_OF}")
print("forecast:", ml.predict_next_month_revenue(txs)["value"])
print("open AR:", summary["open_amount"], "of which overdue:", summary["overdue_amount"])
print("chase queue:")
for f in summary["top"]:
    print(f"  {f['vendor_or_customer']:<14} days_until_due={f['days_until_due']:>4}"
          f"  p_late={f['p_late']:<6}  severity={f['severity']}")


## Note on honesty (SKILL.md §11.4)

Training labels come from a rule **we** wrote, deliberately blurred by the
generator's `NOISE_FLIP_PROB` flips. The F1 ceiling was fixed at 0.85 before any
result was seen; observed values sit well below it and are reported as-is.